## Start spark session (Feature engineering)

In [1]:
from scripts import Spark_Init as spi
spark = spi.start_spark("Data Ingestion")

Library location check:
Spark home: C:\Users\senza\Desktop\Projects\PersonalProjects\PythonProjects\MyMLClassProject\content\spark-3.5.1-bin-hadoop3
Java home: C:\Users\senza\Desktop\Projects\PersonalProjects\PythonProjects\MyMLClassProject\content\java\jdk-17
hadoop home: C:\Users\senza\Desktop\Projects\PersonalProjects\PythonProjects\MyMLClassProject\content\hadoop
Spark Session 'Data Ingestion' is active!
Spark UI available at: http://10.223.136.209:4040


In [2]:
from pyspark.sql.functions import col,month, mean, hour,dayofweek,year,unix_timestamp,isnull,isnan
from pyspark.sql.types import DoubleType
from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.ml import Pipeline

## Load cleaned up data

In [3]:
df = spark.read.parquet("../data/cleaned_data")
df.cache()

DataFrame[VendorID: int, tpep_pickup_datetime: timestamp, tpep_dropoff_datetime: timestamp, passenger_count: double, trip_distance: double, RatecodeID: double, store_and_fwd_flag: string, PULocationID: int, DOLocationID: int, payment_type: int, fare_amount: double, extra: double, mta_tax: double, tip_amount: double, tolls_amount: double, improvement_surcharge: double, total_amount: double, congestion_surcharge: double, Airport_fee: double, pickup_month: int]

## Preprocessing Data

#### 1. Extract meaningful date and time features from 'tpep_pickup_datetime'

In [4]:
# Extract hour, day of week, month, and year
df = df.withColumn("pickup_hour", hour("tpep_pickup_datetime")) \
       .withColumn("pickup_dayofweek", dayofweek("tpep_pickup_datetime")) \
       .withColumn("pickup_month", month("tpep_pickup_datetime")) \
       .withColumn("pickup_year", year("tpep_pickup_datetime"))

# Calculate trip duration (in minutes) using 'tpep_pickup_datetime' and 'tpep_dropoff_datetime'
df = df.withColumn("trip_duration",
                   (unix_timestamp("tpep_dropoff_datetime") - unix_timestamp("tpep_pickup_datetime")) / 60)


#### 2. Interaction Features

In [5]:
# Create features that represent interactions between columns
df = df.withColumn("distance_per_passenger", col("trip_distance") / col("passenger_count"))
df = df.withColumn("fare_per_distance", col("fare_amount") / col("trip_distance"))
df = df.withColumn("total_per_passenger", col("total_amount") / col("passenger_count"))

#### 3. Aggregated Features

In [6]:
# Aggregations based on locations might be useful if there's a logical grouping for PULocationID
location_aggregates = df.groupBy("PULocationID").agg(
    mean("fare_amount").alias("avg_fare_by_location"),
    mean("trip_duration").alias("avg_duration_by_location")
)
df = df.join(location_aggregates, on="PULocationID", how="left")

In [7]:
df.show(6, truncate=False)

+------------+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------+-----------+----------------+-----------+-------------+----------------------+-----------------+-------------------+--------------------+------------------------+
|PULocationID|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|pickup_month|pickup_hour|pickup_dayofweek|pickup_year|trip_duration|distance_per_passenger|fare_per_distance|total_per_passenger|avg_fare_by_location|avg_duration_by_location|
+------------+--------+--------------------+---------------------+---------------+-------------+----------+-

## Check feature imporatnce for fare price prediction

In [8]:
input_features = ["PULocationID", "VendorID", "passenger_count", "trip_distance", "RatecodeID",
                  "DOLocationID", "payment_type", "extra", "mta_tax", "tip_amount",
                  "tolls_amount", "improvement_surcharge", "congestion_surcharge",
                  "Airport_fee", "pickup_hour", "pickup_dayofweek", "pickup_month",
                  "pickup_year", "trip_duration", "distance_per_passenger",
                  "fare_per_distance", "total_per_passenger", "avg_fare_by_location",
                  "avg_duration_by_location"]

for feature in input_features:
    correlation = df.stat.corr("fare_amount", feature)
    print(f"Correlation between fare_amount and {feature}: {correlation}")

Correlation between fare_amount and PULocationID: -0.0504206305107309
Correlation between fare_amount and VendorID: 0.03706383880954421
Correlation between fare_amount and passenger_count: nan
Correlation between fare_amount and trip_distance: 0.8757433418573832
Correlation between fare_amount and RatecodeID: -0.0005197402678899489
Correlation between fare_amount and DOLocationID: -0.0780355984420077
Correlation between fare_amount and payment_type: -0.04123030216589308
Correlation between fare_amount and extra: -0.03758055068948794
Correlation between fare_amount and mta_tax: nan
Correlation between fare_amount and tip_amount: 0.39656910959198155
Correlation between fare_amount and tolls_amount: nan
Correlation between fare_amount and improvement_surcharge: nan
Correlation between fare_amount and congestion_surcharge: nan
Correlation between fare_amount and Airport_fee: nan
Correlation between fare_amount and pickup_hour: 0.0037854736112340475
Correlation between fare_amount and picku

## Define target and selected input features to retain

In [9]:
target_column = "fare_amount"
selected_features = [
    "trip_distance", "tip_amount", "trip_duration", "distance_per_passenger",
    "fare_per_distance", "total_per_passenger", "avg_fare_by_location",
    "avg_duration_by_location"
]

# Retain only the selected features and the target column
columns_to_keep = [target_column] + selected_features
df = df.select(*columns_to_keep)

In [10]:
# Show schema to ascertain dataframe structure
df.printSchema()

root
 |-- fare_amount: double (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- trip_duration: double (nullable = true)
 |-- distance_per_passenger: double (nullable = true)
 |-- fare_per_distance: double (nullable = true)
 |-- total_per_passenger: double (nullable = true)
 |-- avg_fare_by_location: double (nullable = true)
 |-- avg_duration_by_location: double (nullable = true)



## Modeling dataframe for training

In [11]:
# Step 1: Ensure the target column and selected features are cast to DoubleType
for column in [target_column] + selected_features:
    df = df.withColumn(column, col(column).cast(DoubleType()))

# Step 2: Remove rows with NaN, null, or infinite values in selected columns
for column in [target_column] + selected_features:
    df = df.filter(~isnan(col(column)) & ~isnull(col(column)) & (col(column) != float("inf")) & (col(column) != float("-inf")))

## Dataframe transformation to be use in model training notebook

In [12]:
# Step 3: Assemble selected features into a single feature vector
assembler = VectorAssembler(inputCols=selected_features, outputCol="features")

In [13]:
# Step 4: Normalize the feature vector
scaler = StandardScaler(inputCol="features", outputCol="scaled_features", withMean=True, withStd=True)

In [14]:
pipeline = Pipeline(stages=[assembler, scaler])
model = pipeline.fit(df)
df = model.transform(df)

## Manually check dataset meets criteria

In [15]:
df.show(50, truncate=False)

+-----------+-------------+----------+------------------+----------------------+------------------+-------------------+--------------------+------------------------+-------------------------------------------------------------------------------------------------+--------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|fare_amount|trip_distance|tip_amount|trip_duration     |distance_per_passenger|fare_per_distance |total_per_passenger|avg_fare_by_location|avg_duration_by_location|features                                                                                         |scaled_features                                                                                                                                                     |
+-----------+-------------+----------+------------------+----------------------+------------------+-------------------+--------------------+--

In [16]:
df.write.mode("overwrite") \
    .parquet("../data/features_data")